# LeetCode 1529 — Minimum Suffix Flips bằng Python

Link đề: https://leetcode.com/problems/minimum-suffix-flips/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minFlips(self, target: str) -> int:
        ...
```

Bài này còn được biết với tên cũ *Bulb Switcher IV* (gợi ý của đề vẫn nói về "bóng đèn"): mỗi thao tác bật/tắt toàn bộ dãy bóng từ vị trí `i` tới cuối.

## 1. Mục tiêu học

1. Hiểu thao tác **lật hậu tố** (suffix flip) và nhận ra thứ tự thao tác **không quan trọng**, mỗi vị trí chỉ cần lật **tối đa một lần**.
2. Viết được lời giải mô phỏng tham lam `O(n²)` theo gợi ý của đề: duyệt trái → phải, sai thì lật.
3. Tối ưu thành `O(n)` bằng cách **không lật thật** mà chỉ theo dõi **tính chẵn lẻ** số lần đã lật.
4. Rút gọn thành công thức: đáp án = **số lần ký tự thay đổi** khi đọc chuỗi `"0" + target` (đếm "biên" giữa các khối).
5. Chứng minh được tính tối ưu bằng lập luận "mỗi biên cần ít nhất một thao tác", và kiểm chứng bằng vét cạn `2ⁿ` trên chuỗi nhỏ.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `target` độ dài `n` (đánh chỉ số từ 0). Chuỗi `s` cùng độ dài ban đầu **toàn số 0**. Mỗi thao tác: chọn chỉ số `i` (`0 ≤ i < n`) và **lật** mọi bit trong đoạn `[i, n − 1]` (`'0'` ↔ `'1'`). Trả về **số thao tác ít nhất** để `s` trở thành `target`.

**Ví dụ 1:** `target = "10111"` → `3`. Từ `"00000"`: lật từ `i = 2` → `"00111"`, lật từ `i = 0` → `"11000"`, lật từ `i = 1` → `"10111"`.

**Ví dụ 2:** `target = "101"` → `3`. `"000"` → (i=0) `"111"` → (i=1) `"100"` → (i=2) `"101"`.

**Ví dụ 3:** `target = "00000"` → `0` (đã bằng sẵn).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `target[i]` là `'0'` hoặc `'1'`.

In [1]:
import random
import time
from itertools import groupby, product
from typing import List

examples = [("10111", 3), ("101", 3), ("00000", 0)]
for t, want in examples:
    print(f"target = {t!r:9} -> mong đợi {want}")

target = '10111'   -> mong đợi 3
target = '101'     -> mong đợi 3
target = '00000'   -> mong đợi 0


Thử tự tay chạy lại chuỗi thao tác của ví dụ 1 bằng một hàm lật hậu tố nhỏ — để chắc rằng ta hiểu đúng thao tác của đề.

In [2]:
def flip_suffix(s: str, i: int) -> str:
    """Lật mọi bit từ chỉ số i tới cuối."""
    return s[:i] + "".join("1" if c == "0" else "0" for c in s[i:])

s = "00000"
for i in [2, 0, 1]:
    s2 = flip_suffix(s, i)
    print(f"lật từ i={i}: {s} -> {s2}")
    s = s2
print("bằng target?", s == "10111")

lật từ i=2: 00000 -> 00111
lật từ i=0: 00111 -> 11000
lật từ i=1: 11000 -> 10111
bằng target? True


## 3. Kiến thức nền

### 3.1 Chuỗi trong Python là bất biến (immutable)

Không thể gán `s[i] = '1'`. Muốn "sửa" phải tạo chuỗi mới (cắt lát + nối) hoặc đổi sang `list` rồi sửa từng phần tử. Mỗi lần tạo chuỗi mới tốn `O(n)` — đây là nguồn gốc độ phức tạp `O(n²)` của cách mô phỏng.

In [3]:
s = "000"
try:
    s[1] = "1"
except TypeError as e:
    print("Không gán được:", e)
arr = list(s)
arr[1] = "1"
print("Dùng list:", "".join(arr))
print("Dùng cắt lát:", s[:1] + "1" + s[2:])

Không gán được: 'str' object does not support item assignment
Dùng list: 010
Dùng cắt lát: 010


### 3.2 Phép XOR để lật bit

Với số nguyên 0/1: `b ^ 1` lật bit. Lật hai lần thì về như cũ: `b ^ 1 ^ 1 == b`. Đây là lý do **lật cùng một vị trí hai lần là vô ích**. Với ký tự, đổi sang số bằng `int(c)` hoặc `ord(c) - 48`.

In [4]:
for b in [0, 1]:
    print(f"b={b}: b^1={b ^ 1}, b^1^1={b ^ 1 ^ 1}")
print([int(c) for c in "1011"], [ord(c) - 48 for c in "1011"])

b=0: b^1=1, b^1^1=0
b=1: b^1=0, b^1^1=1
[1, 0, 1, 1] [1, 0, 1, 1]


### 3.3 `zip` để duyệt từng cặp kề nhau

`zip(a, a[1:])` sinh các cặp `(a[i], a[i+1])`. Ghép thêm `"0"` vào đầu: `zip("0" + t, t)` cho cặp (ký tự trước, ký tự hiện tại), trong đó "ký tự trước" của vị trí 0 là `'0'` — đúng với trạng thái ban đầu của `s`.

In [5]:
t = "10111"
print(list(zip("0" + t, t)))
print("số cặp khác nhau:", sum(a != b for a, b in zip("0" + t, t)))

[('0', '1'), ('1', '0'), ('0', '1'), ('1', '1'), ('1', '1')]
số cặp khác nhau: 3


### 3.4 `itertools.groupby` — gom các khối ký tự liên tiếp giống nhau

`groupby("0011100")` trả về từng khối `('0', ...), ('1', ...), ('0', ...)`. Số khối giúp đếm số "biên" giữa các khối: `số biên = số khối − 1`.

In [6]:
for key, grp in groupby("0011100"):
    print(key, list(grp))
print("số khối:", sum(1 for _ in groupby("0011100")))

0 ['0', '0']
1 ['1', '1', '1']
0 ['0', '0']
số khối: 3


### 3.5 `itertools.product` — sinh mọi tổ hợp 0/1

`product([0, 1], repeat=n)` sinh `2ⁿ` bộ — dùng để **vét cạn** mọi tập vị trí được lật (kiểm chứng tính tối ưu trên `n` nhỏ).

In [7]:
for bits in product([0, 1], repeat=3):
    print(bits, end="  ")
print()

(0, 0, 0)  (0, 0, 1)  (0, 1, 0)  (0, 1, 1)  (1, 0, 0)  (1, 0, 1)  (1, 1, 0)  (1, 1, 1)  


## 4. Ý tưởng nền tảng / trực giác

Làm tay với `target = "10111"`. Ta đi từ trái sang phải và giữ chuỗi `s` hiện tại:

| i | s trước | s[i] | target[i] | Hành động | s sau |
|---|---------|------|-----------|-----------|-------|
| 0 | 00000 | 0 | 1 | khác → lật từ 0 | 11111 |
| 1 | 11111 | 1 | 0 | khác → lật từ 1 | 10000 |
| 2 | 10000 | 0 | 1 | khác → lật từ 2 | 10111 |
| 3 | 10111 | 1 | 1 | giống → bỏ qua | 10111 |
| 4 | 10111 | 1 | 1 | giống → bỏ qua | 10111 |

Tổng 3 lần lật. Hai quan sát quan trọng:

- Lật từ `i` **không đụng** tới các vị trí `< i`. Nên khi đã xử lý xong bên trái thì bên trái không bao giờ bị phá nữa.
- Vị trí `0` chỉ bị ảnh hưởng bởi thao tác "lật từ 0". Vậy `s[0]` sai thì **bắt buộc** phải lật từ 0. Lập luận tương tự lan dần sang phải → chiến lược tham lam là **duy nhất** (với mỗi vị trí, việc lật hay không bị ép buộc).

Nhìn kỹ cột "Hành động": ta lật đúng tại những vị trí mà `target[i] != target[i-1]` (coi `target[-1] = '0'`). Ở ví dụ: `0|1 0 1 1 1` → các biên tại `i = 0, 1, 2`.

In [8]:
t = "10111"
prev = "0"
for i, c in enumerate(t):
    print(f"i={i}: trước={prev} hiện tại={c} -> {'BIÊN (cần lật)' if c != prev else 'giữ nguyên'}")
    prev = c

i=0: trước=0 hiện tại=1 -> BIÊN (cần lật)
i=1: trước=1 hiện tại=0 -> BIÊN (cần lật)
i=2: trước=0 hiện tại=1 -> BIÊN (cần lật)
i=3: trước=1 hiện tại=1 -> giữ nguyên
i=4: trước=1 hiện tại=1 -> giữ nguyên


## 5. Cách 1 — Brute Force / baseline

### 5a. Vét cạn mọi tập vị trí lật (chỉ để kiểm chứng)

Vì lật hai lần = không lật và thứ tự lật không ảnh hưởng kết quả (mỗi bit `j` bị lật đúng bằng số thao tác có `i ≤ j`, phép cộng giao hoán), một lời giải được mô tả bằng **tập** vị trí lật. Thử cả `2ⁿ` tập, giữ tập nhỏ nhất cho ra `target`.

In [9]:
def brute_subsets(target: str) -> int:
    n = len(target)
    best = None
    for choose in product([0, 1], repeat=n):
        flips = 0          # số lần lật đã áp dụng lên vị trí j (tính chẵn lẻ)
        ok = True
        for j in range(n):
            flips ^= choose[j]               # lật từ j ảnh hưởng mọi vị trí >= j
            if str(flips) != target[j]:
                ok = False
                break
        if ok:
            k = sum(choose)
            best = k if best is None else min(best, k)
    return best

for t, want in examples:
    print(t, brute_subsets(t), "mong đợi", want)

10111 3 mong đợi 3
101 3 mong đợi 3
00000 0 mong đợi 0


Độ phức tạp `O(2ⁿ · n)` — chỉ dùng được với `n ≤ ~16`. Điều thú vị: hàm trên luôn tìm thấy **đúng một** tập hợp lệ (ta sẽ thấy ở mục 10). Đây là gợi ý mạnh rằng lời giải bị "ép buộc".

In [10]:
def count_valid_sets(target: str) -> int:
    n, cnt = len(target), 0
    for choose in product([0, 1], repeat=n):
        f, ok = 0, True
        for j in range(n):
            f ^= choose[j]
            if str(f) != target[j]:
                ok = False
                break
        cnt += ok
    return cnt

print({t: count_valid_sets(t) for t in ["10111", "101", "0000", "0110"]})

{'10111': 1, '101': 1, '0000': 1, '0110': 1}


### 5b. Mô phỏng tham lam lật thật — `O(n²)`

Đúng theo gợi ý của đề: duyệt `i` tăng dần, nếu `s[i] != target[i]` thì lật hậu tố từ `i` (lật thật trên list). Đúng nhưng mỗi lần lật tốn `O(n)`.

In [11]:
def simulate(target: str) -> int:
    n = len(target)
    s = ["0"] * n
    ops = 0
    for i in range(n):
        if s[i] != target[i]:
            ops += 1
            for j in range(i, n):                # lật thật đoạn [i, n-1]
                s[j] = "1" if s[j] == "0" else "0"
    assert "".join(s) == target
    return ops

for t, want in examples:
    print(t, simulate(t), "mong đợi", want)

10111 3 mong đợi 3
101 3 mong đợi 3
00000 0 mong đợi 0


**Độ phức tạp:** worst case (ví dụ `"1010...10"`) phải lật ở mọi vị trí → `n + (n−1) + … + 1 = O(n²)`; với `n = 10⁵` khoảng `5·10⁹` phép gán — quá chậm. Space `O(n)`.

**Nút thắt:** ta lật **thật** cả hậu tố, trong khi lúc xét vị trí `i` ta chỉ cần biết **giá trị hiện tại của `s[i]`**, tức là "`s[i]` đã bị lật chẵn hay lẻ lần".

In [12]:
worst = "10" * 1500
t0 = time.perf_counter(); r = simulate(worst); t1 = time.perf_counter()
print(f"n={len(worst)}: {r} thao tác, mất {t1 - t0:.3f}s (n=10^5 sẽ chậm hơn ~1000 lần)")

n=3000: 3000 thao tác, mất 0.201s (n=10^5 sẽ chậm hơn ~1000 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — Chỉ cần tính chẵn lẻ.** Mọi thao tác lật từ `i' ≤ i` đều lật `s[i]`. Nên giá trị hiện tại của `s[i]` = `(số thao tác đã làm) mod 2`. Gọi `cur` là bit đó: ban đầu `cur = 0`, mỗi lần lật thì `cur ^= 1`.

- **Nếu** `cur != target[i]` **thì** lật (`ops += 1`, `cur ^= 1`).
- **Nếu** `cur == target[i]` **thì** bỏ qua.

**Insight 2 — `cur` sau bước `i` chính là `target[i]`.** Sau khi xử lý vị trí `i`, ta luôn có `cur == target[i]`. Vậy ở bước `i+1`, điều kiện "lật" trở thành `target[i] != target[i+1]`.

**Invariant:** trước khi xét vị trí `i`, `cur = target[i−1]` (quy ước `target[−1] = '0'`) và mọi vị trí `< i` đã khớp.

⇒ **Đáp án = số chỉ số `i` mà `target[i] != target[i−1]`, với `target[−1] = '0'`** = số "biên" của chuỗi `"0" + target`.

## 7. Các cách tiếp theo

### Cách 2 — Tham lam theo dõi tính chẵn lẻ, `O(n)`

Thay vòng lặp lật hậu tố bằng một biến `cur`. Giải thích từng dòng:

- `cur = 0`: bit hiện tại mà mọi vị trí chưa xét "đang có" (do các lần lật trước).
- `int(c) != cur`: vị trí này đang sai → lật hậu tố từ đây; điều duy nhất thay đổi đối với phần chưa xét là `cur` đảo.

In [13]:
def parity_greedy(target: str) -> int:
    cur = 0          # giá trị hiện tại của mọi bit chưa xét
    ops = 0
    for c in target:
        if int(c) != cur:
            ops += 1
            cur ^= 1
    return ops

for t, want in examples:
    print(t, parity_greedy(t), "mong đợi", want)

10111 3 mong đợi 3
101 3 mong đợi 3
00000 0 mong đợi 0


Code nộp LeetCode cho cách 2:

In [14]:
class SolutionParity:
    def minFlips(self, target: str) -> int:
        cur, ops = "0", 0
        for c in target:
            if c != cur:
                ops += 1
                cur = c          # sau khi lật, bit hiện tại khớp với c
        return ops

print([SolutionParity().minFlips(t) for t, _ in examples])

[3, 3, 0]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — Đếm số biên bằng `zip`, một dòng

Từ insight 2, đáp án chỉ là số cặp kề khác nhau của `"0" + target`. Dùng `zip` để duyệt cặp và `sum` các giá trị bool (`True` = 1).

In [15]:
def count_boundaries(target: str) -> int:
    return sum(a != b for a, b in zip("0" + target, target))

for t, want in examples:
    print(t, count_boundaries(t), "mong đợi", want)

10111 3 mong đợi 3
101 3 mong đợi 3
00000 0 mong đợi 0


In [16]:
class SolutionZip:
    def minFlips(self, target: str) -> int:
        return sum(a != b for a, b in zip("0" + target, target))

print([SolutionZip().minFlips(t) for t, _ in examples])

[3, 3, 0]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` vì tạo chuỗi `"0" + target` (có thể tránh bằng chỉ số, nhưng thực tế không đáng kể).

### Cách 4 — Đếm số khối bằng `groupby` / `str.find`

Chuỗi `target` gồm các khối liên tiếp giống nhau. Số biên trong `"0" + target` = (số khối của `target`) − 1, cộng thêm 1 nếu khối đầu là `'1'` (biên giữa `'0'` ảo và khối đầu). Một cách nghĩ khác: bỏ các số 0 ở đầu (chúng không cần lật), rồi đếm số khối còn lại.

In [17]:
def count_blocks(target: str) -> int:
    stripped = target.lstrip("0")         # các số 0 đầu đã khớp sẵn
    return sum(1 for _ in groupby(stripped))

for t, want in examples + [("0011100", 2), ("1", 1), ("0", 0)]:
    print(f"{t!r:10} khối sau lstrip = {[k for k, _ in groupby(t.lstrip('0'))]} -> {count_blocks(t)} (mong đợi {want})")

'10111'    khối sau lstrip = ['1', '0', '1'] -> 3 (mong đợi 3)
'101'      khối sau lstrip = ['1', '0', '1'] -> 3 (mong đợi 3)
'00000'    khối sau lstrip = [] -> 0 (mong đợi 0)
'0011100'  khối sau lstrip = ['1', '0'] -> 2 (mong đợi 2)
'1'        khối sau lstrip = ['1'] -> 1 (mong đợi 1)
'0'        khối sau lstrip = [] -> 0 (mong đợi 0)


In [18]:
class SolutionGroupby:
    def minFlips(self, target: str) -> int:
        return sum(1 for _ in groupby(target.lstrip("0")))

print([SolutionGroupby().minFlips(t) for t, _ in examples])

[3, 3, 0]


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (do `lstrip` tạo chuỗi mới).

Một biến thể "thuần chuỗi" khác: số biên = số lần xuất hiện của mẫu `"01"` cộng số lần xuất hiện của mẫu `"10"` trong `"0" + target`. Lưu ý `str.count` đếm **không chồng lấn**, nhưng một mẫu 2 ký tự khác nhau như `"01"` không thể chồng lấn với chính nó, nên đếm riêng từng mẫu vẫn chính xác.

In [19]:
def count_patterns(target: str) -> int:
    s = "0" + target
    return s.count("01") + s.count("10")

for t in ["10111", "101", "00000", "010101", "1100"]:
    print(t, count_patterns(t), count_boundaries(t))

10111 3 3
101 3 3
00000 0 0
010101 5 5
1100 2 2


## 8. TRACE từng bước

Bảng trạng thái cho `target = "1100101"` với cách 2 (theo dõi `cur`):

| i | target[i] | cur trước | lật? | ops | cur sau |
|---|-----------|-----------|------|-----|---------|
| 0 | 1 | 0 | có | 1 | 1 |
| 1 | 1 | 1 | không | 1 | 1 |
| 2 | 0 | 1 | có | 2 | 0 |
| 3 | 0 | 0 | không | 2 | 0 |
| 4 | 1 | 0 | có | 3 | 1 |
| 5 | 0 | 1 | có | 4 | 0 |
| 6 | 1 | 0 | có | 5 | 1 |

Hàm `trace_parity` in lại bảng trên, kèm chuỗi `s` **thật** (lật thật song song) để đối chiếu rằng `cur` đúng là giá trị của các bit chưa xét.

In [20]:
def trace_parity(target: str) -> int:
    n = len(target)
    s = ["0"] * n
    cur, ops = 0, 0
    print(f"{'i':>2} | t[i] | cur trước | lật? | ops | cur sau | s thật sau bước")
    for i, c in enumerate(target):
        before = cur
        flip = int(c) != cur
        if flip:
            ops += 1
            cur ^= 1
            for j in range(i, n):
                s[j] = "1" if s[j] == "0" else "0"
        assert all(int(x) == cur for x in s[i:]), "cur phải bằng mọi bit chưa xét"
        print(f"{i:>2} |  {c}   |     {before}     | {'có ' if flip else 'không'} | {ops:>3} |    {cur}    | {''.join(s)}")
    return ops

print("Kết quả:", trace_parity("1100101"))

 i | t[i] | cur trước | lật? | ops | cur sau | s thật sau bước
 0 |  1   |     0     | có  |   1 |    1    | 1111111
 1 |  1   |     1     | không |   1 |    1    | 1111111
 2 |  0   |     1     | có  |   2 |    0    | 1100000
 3 |  0   |     0     | không |   2 |    0    | 1100000
 4 |  1   |     0     | có  |   3 |    1    | 1100111
 5 |  0   |     1     | có  |   4 |    0    | 1100100
 6 |  1   |     0     | có  |   5 |    1    | 1100101
Kết quả: 5


Trace thêm cách 3 (đếm biên) trên ví dụ 1 để thấy cùng các vị trí lật:

In [21]:
def trace_boundaries(target: str) -> int:
    total = 0
    for i, (a, b) in enumerate(zip("0" + target, target)):
        total += a != b
        print(f"i={i}: cặp ({a},{b}) {'khác -> +1' if a != b else 'giống'} | tổng = {total}")
    return total

print("Kết quả:", trace_boundaries("10111"))

i=0: cặp (0,1) khác -> +1 | tổng = 1
i=1: cặp (1,0) khác -> +1 | tổng = 2
i=2: cặp (0,1) khác -> +1 | tổng = 3
i=3: cặp (1,1) giống | tổng = 3
i=4: cặp (1,1) giống | tổng = 3
Kết quả: 3


## 9. Code nộp LeetCode

Bản nộp chính (cách 2/3 — `O(n)` thời gian, `O(1)` bộ nhớ), đúng tên class `Solution` và method `minFlips(self, target: str) -> int`:

In [22]:
class Solution:
    def minFlips(self, target: str) -> int:
        ops = 0
        prev = "0"                 # trạng thái ban đầu của s
        for c in target:
            if c != prev:          # gặp biên -> bắt buộc lật một lần ở đây
                ops += 1
                prev = c
        return ops

for t, want in examples:
    got = Solution().minFlips(t)
    print(t, got, "OK" if got == want else "SAI")

10111 3 OK
101 3 OK
00000 0 OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (thứ tự & lặp lại không quan trọng).** Bit cuối cùng ở vị trí `j` = (số thao tác có chỉ số `i ≤ j`) mod 2. Giá trị này không phụ thuộc thứ tự thao tác; và hai thao tác cùng chỉ số triệt tiêu nhau. Vậy mọi lời giải tối ưu dùng mỗi chỉ số **tối đa một lần**, mô tả bằng vector `x[0..n−1] ∈ {0,1}`.

**Bổ đề 2 (lời giải bị ép buộc).** Điều kiện `s = target` ⇔ với mọi `j`: `x[0] ⊕ x[1] ⊕ … ⊕ x[j] = target[j]`. Lấy hiệu hai đẳng thức liên tiếp: `x[j] = target[j] ⊕ target[j−1]` (với `target[−1] = 0`). Vậy vector `x` **duy nhất** — chẳng có lựa chọn nào khác!

**Kết luận.** Số thao tác = `Σ x[j]` = số `j` mà `target[j] != target[j−1]` = số biên. Vì lời giải (không lặp chỉ số) là duy nhất, nó cũng là tối thiểu; lời giải có lặp chỉ số chỉ tốn thêm. Đó cũng là lý do `count_valid_sets` ở mục 5 luôn ra `1`.

**Chặn dưới trực quan:** mỗi biên `target[j−1] != target[j]` đòi hỏi bit `j−1` và bit `j` bị lật số lần khác tính chẵn lẻ, nên **phải có** thao tác bắt đầu đúng tại `j`. Các biên khác nhau cần các thao tác khác nhau ⇒ cần ≥ số biên thao tác; tham lam dùng đúng bằng số đó.

In [23]:
# Kiểm chứng công thức x[j] = t[j] xor t[j-1] tái tạo đúng target
def ops_vector(target):
    prev, x = 0, []
    for c in target:
        x.append(int(c) ^ prev)
        prev = int(c)
    return x

t = "1100101"
x = ops_vector(t)
s = "0" * len(t)
for i, xi in enumerate(x):
    if xi:
        s = flip_suffix(s, i)
print("x =", x, "| áp dụng ra", s, "| khớp:", s == t, "| số thao tác:", sum(x))

x = [1, 0, 1, 0, 1, 1, 1] | áp dụng ra 1100101 | khớp: True | số thao tác: 5


## 11. Phân tích Time / Space

Gọi `n = len(target)`.

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| Vét cạn tập con | `O(2ⁿ · n)` | `O(n)` | chỉ để kiểm chứng |
| Mô phỏng lật thật | `O(n²)` worst, `O(n)` best (ít biên) | `O(n)` | worst khi chuỗi xen kẽ `1010…` |
| Theo dõi chẵn lẻ | `O(n)` | `O(1)` | bản nộp chính |
| Đếm biên `zip` | `O(n)` | `O(n)` (chuỗi ghép) | gọn nhất |
| `groupby` khối | `O(n)` | `O(n)` | cách nhìn "khối" |

Không có khác biệt average/worst cho các cách `O(n)`: luôn duyệt hết chuỗi đúng một lần.

## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, toàn 0, toàn 1, xen kẽ), đối chiếu mọi cách với vét cạn trên **mọi** chuỗi độ dài ≤ 10, random seed cố định với `n` vừa, và test lớn `n = 10⁵`.

In [24]:
METHODS = {
    "simulate": simulate,
    "parity": parity_greedy,
    "SolutionParity": lambda t: SolutionParity().minFlips(t),
    "zip": lambda t: SolutionZip().minFlips(t),
    "groupby": lambda t: SolutionGroupby().minFlips(t),
    "patterns": count_patterns,
    "Solution": lambda t: Solution().minFlips(t),
}
edge = [("0", 0), ("1", 1), ("01", 1), ("10", 2), ("11", 1), ("1111", 1),
        ("0001", 1), ("1000", 2), ("010101", 5), ("101010", 6), ("0011100", 2)]
for t, want in examples + edge:
    assert brute_subsets(t) == want, t
    for name, f in METHODS.items():
        assert f(t) == want, (name, t, f(t), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [25]:
cnt = 0
for n in range(1, 11):
    for bits in product("01", repeat=n):
        t = "".join(bits)
        ref = brute_subsets(t) if n <= 8 else count_boundaries(t)
        for name, f in METHODS.items():
            assert f(t) == ref, (name, t)
        cnt += 1
print(f"Đã kiểm tra toàn bộ {cnt} chuỗi độ dài 1..10 (vét cạn tới n=8): OK")

Đã kiểm tra toàn bộ 2046 chuỗi độ dài 1..10 (vét cạn tới n=8): OK


In [26]:
rng = random.Random(1529)
for _ in range(300):
    n = rng.randint(1, 300)
    p = rng.choice([0.1, 0.5, 0.9])
    t = "".join("1" if rng.random() < p else "0" for _ in range(n))
    ref = simulate(t)
    for name, f in METHODS.items():
        assert f(t) == ref, (name, t)
print("300 chuỗi ngẫu nhiên (seed 1529): mọi cách khớp mô phỏng")

300 chuỗi ngẫu nhiên (seed 1529): mọi cách khớp mô phỏng


In [27]:
rng2 = random.Random(7)
big_rand = "".join(rng2.choice("01") for _ in range(10**5))
big_alt = "10" * 50000
big_one = "1" * 10**5
for name in ["parity", "zip", "groupby", "Solution"]:
    t0 = time.perf_counter()
    res = [METHODS[name](b) for b in (big_rand, big_alt, big_one)]
    print(f"{name:9s}: {res} ({time.perf_counter() - t0:.3f}s)")
    assert res[1] == 10**5 and res[2] == 1

parity   : [49792, 100000, 1] (0.020s)


zip      : [49792, 100000, 1] (0.013s)
groupby  : [49792, 100000, 1] (0.012s)
Solution : [49792, 100000, 1] (0.008s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm số chữ `'1'` thay vì số biên

Nghĩ rằng "mỗi bit 1 cần một lần lật". Sai vì một lần lật hậu tố có thể tạo ra **cả khối** số 1, và các số 0 nằm sau số 1 cũng cần lật.

In [28]:
def wrong_count_ones(target):
    return target.count("1")        # SAI

for t, want in examples + [("110", 2), ("1111", 1)]:
    print(f"{t!r:8} sai: {wrong_count_ones(t)} | đúng: {want}")

'10111'  sai: 4 | đúng: 3
'101'    sai: 2 | đúng: 3
'00000'  sai: 0 | đúng: 0
'110'    sai: 2 | đúng: 2
'1111'   sai: 4 | đúng: 1


Sửa: đếm số vị trí `i` mà `target[i] != target[i−1]` với `target[−1] = '0'` (hàm `Solution` ở mục 9).

### Lỗi 2 — Khởi tạo `prev = target[0]` (quên trạng thái ban đầu toàn 0)

Nếu khởi tạo `prev` bằng ký tự đầu, ta bỏ sót thao tác cần thiết khi `target` bắt đầu bằng `'1'`.

In [29]:
def wrong_prev_init(target):
    prev, ops = target[0], 0        # SAI: phải là "0"
    for c in target:
        if c != prev:
            ops += 1
            prev = c
    return ops

for t, want in examples + [("1", 1), ("0110", 2)]:
    print(f"{t!r:8} sai: {wrong_prev_init(t)} | đúng: {want}")

'10111'  sai: 2 | đúng: 3
'101'    sai: 2 | đúng: 3
'00000'  sai: 0 | đúng: 0
'1'      sai: 0 | đúng: 1
'0110'   sai: 2 | đúng: 2


Sửa: `prev = "0"`. Quy tắc nhớ: "ký tự đứng trước vị trí 0" chính là trạng thái ban đầu của `s`.

### Lỗi 3 — Nhân đôi số khối `'1'`

Lập luận "mỗi khối 1 cần lật lên rồi lật xuống" → `2 × số khối 1`. Sai với khối 1 ở **cuối** chuỗi: không cần lật xuống nữa.

In [30]:
def wrong_double_blocks(target):
    blocks_of_one = sum(1 for k, _ in groupby(target) if k == "1")
    return 2 * blocks_of_one        # SAI khi chuỗi kết thúc bằng '1'

for t, want in examples + [("0110", 2), ("1", 1)]:
    print(f"{t!r:8} sai: {wrong_double_blocks(t)} | đúng: {want}")

'10111'  sai: 4 | đúng: 3
'101'    sai: 4 | đúng: 3
'00000'  sai: 0 | đúng: 0
'0110'   sai: 2 | đúng: 2
'1'      sai: 2 | đúng: 1


Sửa: `2 × số khối 1 − (1 nếu target kết thúc bằng '1')` — công thức này **đúng** và tương đương đếm biên:

In [31]:
def fixed_double_blocks(target):
    blocks_of_one = sum(1 for k, _ in groupby(target) if k == "1")
    return 2 * blocks_of_one - (target[-1] == "1")

print([fixed_double_blocks(t) for t, _ in examples + [("0110", 2), ("1", 1)]])

[3, 3, 0, 2, 1]


### Lỗi 4 — Mô phỏng nhưng chỉ lật **một bit** thay vì cả hậu tố

Quên rằng thao tác lật đến tận cuối chuỗi → thực chất lại đếm số bit 1, và chuỗi cuối cùng không bằng `target` theo luật của đề.

In [32]:
def wrong_flip_one(target):
    s = ["0"] * len(target)
    ops = 0
    for i in range(len(target)):
        if s[i] != target[i]:
            ops += 1
            s[i] = target[i]        # SAI: chỉ sửa đúng 1 bit
    return ops

for t, want in examples:
    print(f"{t!r:8} sai: {wrong_flip_one(t)} | đúng: {want}")

'10111'  sai: 4 | đúng: 3
'101'    sai: 2 | đúng: 3
'00000'  sai: 0 | đúng: 0


Sửa: lật `s[j]` cho mọi `j ≥ i` (hàm `simulate`), hoặc tốt hơn dùng biến `cur`.

### Lỗi 5 — Mô phỏng bằng nối chuỗi trong vòng lặp → quá thời gian

Kết quả đúng nhưng `O(n²)`; chuỗi xen kẽ `n = 10⁵` sẽ TLE. Đo nhanh để thấy chênh lệch:

In [33]:
def slow_string_sim(target):
    s, ops = "0" * len(target), 0
    for i in range(len(target)):
        if s[i] != target[i]:
            ops += 1
            s = flip_suffix(s, i)
    return ops

t = "10" * 2000
t0 = time.perf_counter(); a = slow_string_sim(t); t1 = time.perf_counter()
b = Solution().minFlips(t); t2 = time.perf_counter()
print(f"chậm: {a} ({t1 - t0:.3f}s) | nhanh: {b} ({t2 - t1:.5f}s)")

chậm: 4000 (0.323s) | nhanh: 4000 (0.00020s)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Thao tác ảnh hưởng một hậu tố/tiền tố** + duyệt theo một chiều ⇒ nghĩ ngay tới **tham lam trái → phải** và "phần đã xử lý không bị phá".
- **Thao tác lật (XOR)**: lật hai lần = không lật, thứ tự không quan trọng ⇒ chỉ cần **tính chẵn lẻ** số lần lật, thay mô phỏng bằng một biến `cur`.
- Khi đáp án phụ thuộc "chỗ ký tự đổi giá trị" ⇒ **đếm biên**: `sum(a != b for a, b in zip(prev_seq, seq))`.
- Mẹo trạng thái ban đầu: **ghép một phần tử giả** (`"0" + target`) để xử lý vị trí đầu như mọi vị trí khác.
- Nhận dạng cùng họ: *Minimum Number of K Consecutive Bit Flips* (995), *Bulb Switcher* (319), *Flip String to Monotone Increasing* (926), *Minimum Operations to Make the Array Alternating*.

## 15. Biến thể của bài

1. **Lật tiền tố** `[0, i]` thay vì hậu tố: đối xứng — duyệt từ phải sang trái, đáp án = số biên của `target[::-1]` với phần tử giả `'0'`, tức `sum(a != b for a, b in zip(target[1:] + "0", target))`.
2. **`s` ban đầu bất kỳ** (không phải toàn 0): xét `d[i] = s[i] XOR target[i]` — cần lật đúng những chỗ `d` đổi giá trị; đáp án = số biên của `"0" + d`.
3. **Lật đoạn độ dài cố định `k`** (LeetCode 995): không còn "lật tới cuối" nên cần hàng đợi/mảng hiệu để biết vị trí `i` đang bị bao nhiêu đoạn phủ.
4. **Trả về danh sách các chỉ số cần lật**: chính là các vị trí biên.
5. **Bảng chữ lớn hơn (tăng 1 mod m cho hậu tố)**: số thao tác = `Σ (target[i] − target[i−1]) mod m` — tổng quát hoá của ý tưởng "hiệu kề nhau".

In [34]:
def variant_prefix_flip(target):
    # lật [0, i]: phần tử giả '0' đứng SAU cuối chuỗi
    return sum(a != b for a, b in zip(target, target[1:] + "0"))

def brute_prefix_flip(target):
    n, best = len(target), None
    for choose in product([0, 1], repeat=n):
        ok, f = True, 0
        for j in range(n - 1, -1, -1):
            f ^= choose[j]
            if str(f) != target[j]:
                ok = False
                break
        if ok:
            best = sum(choose) if best is None else min(best, sum(choose))
    return best

for t in ["10111", "101", "00000", "0110", "1"]:
    print(t, variant_prefix_flip(t), brute_prefix_flip(t))

10111 3 3
101 3 3
00000 0 0
0110 2 2
1 1 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1a. Vét cạn tập con | thử mọi tập vị trí lật | `O(2ⁿ·n)` | `O(n)` | kiểm chứng trên `n` nhỏ |
| 1b. Mô phỏng lật thật | tham lam, lật cả hậu tố | `O(n²)` | `O(n)` | hiểu bài, chuẩn đối chiếu |
| 2. Chẵn lẻ `cur` | chỉ nhớ bit hiện tại của phần chưa xét | `O(n)` | `O(1)` | bản nộp chính, dễ giải thích |
| 3. Đếm biên `zip` | số cặp kề khác nhau của `"0"+t` | `O(n)` | `O(n)` | code ngắn nhất |
| 4. `groupby` khối | số khối sau khi bỏ số 0 đầu | `O(n)` | `O(n)` | cách nhìn theo "khối" |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minFlips("0110")` trả về bao nhiêu? Viết số bạn đoán rồi mới chạy ở mục 19.

**Bài 2 (tính tay):** Với `target = "1100101"`, liệt kê các chỉ số cần lật theo thứ tự tham lam và tổng số thao tác.

**Bài 3 (điền TODO):** Hoàn thiện hàm `ex3_parity` theo cách theo dõi chẵn lẻ (dùng số nguyên `cur ∈ {0,1}` và phép `^`).

**Bài 4 (tự viết lại):** Viết `ex4_ops(target)` trả về **danh sách chỉ số** cần lật (theo thứ tự tăng) — áp dụng lần lượt lên chuỗi toàn 0 phải ra `target`.

**Bài 5 (mini challenge):** Chuỗi `s` ban đầu **bất kỳ** (cùng độ dài với `target`). Viết `ex5_min_flips_from(s, target)` trả về số thao tác lật hậu tố ít nhất.

In [35]:
# Bài 3 — điền TODO (cell này chỉ định nghĩa, KHÔNG gọi)
def ex3_parity(target: str) -> int:
    cur = 0
    ops = 0
    for c in target:
        # TODO: nếu bit c khác cur thì tăng ops và lật cur
        pass
    return ops

In [36]:
# Bài 4 & 5 — khung để tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_ops(target: str) -> List[int]:
    # TODO: trả về các chỉ số i cần lật
    raise NotImplementedError

def ex5_min_flips_from(s: str, target: str) -> int:
    # TODO: s ban đầu bất kỳ
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ghép `'0'` vào đầu. (2) Đếm cặp kề khác nhau trong `"00110"`. (3) Các cặp: `00, 01, 11, 10` → hai cặp khác.

**Bài 2.** (1) Duyệt trái → phải với `prev = '0'`. (2) Ghi lại chỉ số mỗi khi `target[i] != prev`. (3) Các biên ở `0, 2, 4, 5, 6`.

**Bài 3.** (1) Đổi `c` sang số bằng `int(c)`. (2) Điều kiện lật: `int(c) != cur`. (3) Trong nhánh lật: `ops += 1; cur ^= 1`.

**Bài 4.** (1) Đáp án là các vị trí biên. (2) Dùng `enumerate(zip("0" + t, t))`. (3) Kiểm tra bằng `flip_suffix` áp dụng lần lượt.

**Bài 5.** (1) Lật hậu tố từ `i` cũng lật bit "sai lệch" `d[j] = s[j] ⊕ target[j]` với mọi `j ≥ i`. (2) Bài toán trở thành: biến `d` thành toàn 0 — tương đương biến chuỗi 0 thành `d`. (3) Đáp án = `Solution().minFlips(d)`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án đều có `assert` kiểm tra (bài 4, 5 đối chiếu với vét cạn trên mọi chuỗi nhỏ).

In [37]:
# Bài 1
ans1 = Solution().minFlips("0110")
print("Bài 1:", ans1)
assert ans1 == 2

# Bài 2
t = "1100101"
ans2 = [i for i, (a, b) in enumerate(zip("0" + t, t)) if a != b]
print("Bài 2:", ans2, "-> số thao tác", len(ans2))
assert ans2 == [0, 2, 4, 5, 6] and len(ans2) == Solution().minFlips(t)

Bài 1: 2
Bài 2: [0, 2, 4, 5, 6] -> số thao tác 5


In [38]:
# Bài 3
def ex3_parity_sol(target: str) -> int:
    cur, ops = 0, 0
    for c in target:
        if int(c) != cur:
            ops += 1
            cur ^= 1
    return ops

rng = random.Random(3)
for _ in range(200):
    t = "".join(rng.choice("01") for _ in range(rng.randint(1, 50)))
    assert ex3_parity_sol(t) == Solution().minFlips(t)
print("Bài 3: OK")

Bài 3: OK


In [39]:
# Bài 4
def ex4_ops_sol(target: str) -> List[int]:
    return [i for i, (a, b) in enumerate(zip("0" + target, target)) if a != b]

for n in range(1, 9):
    for bits in product("01", repeat=n):
        t = "".join(bits)
        s = "0" * n
        idx = ex4_ops_sol(t)
        for i in idx:
            s = flip_suffix(s, i)
        assert s == t and len(idx) == brute_subsets(t)
print("Bài 4: OK, ví dụ", ex4_ops_sol("10111"))

Bài 4: OK, ví dụ [0, 1, 2]


In [40]:
# Bài 5
def ex5_sol(s: str, target: str) -> int:
    d = "".join("1" if a != b else "0" for a, b in zip(s, target))
    return Solution().minFlips(d)

def ex5_brute(s, target):
    n, best = len(s), None
    for choose in product([0, 1], repeat=n):
        cur = s
        for i in range(n):
            if choose[i]:
                cur = flip_suffix(cur, i)
        if cur == target:
            best = sum(choose) if best is None else min(best, sum(choose))
    return best

for n in range(1, 6):
    for a in product("01", repeat=n):
        for b in product("01", repeat=n):
            s, t = "".join(a), "".join(b)
            assert ex5_sol(s, t) == ex5_brute(s, t)
print("Bài 5: OK, ví dụ", ex5_sol("110", "011"))

Bài 5: OK, ví dụ 3


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def minFlips(self, target: str) -> int:
        ops, prev = 0, "0"          # prev = trạng thái ban đầu
        for c in target:
            if c != prev:           # biên -> 1 thao tác
                ops += 1
                prev = c
        return ops
# một dòng: sum(a != b for a, b in zip("0" + target, target))
```

**Template "thao tác lật hậu tố/tiền tố":** duyệt theo hướng mà thao tác *không phá* phần đã xử lý; thay việc lật thật bằng biến chẵn lẻ `cur`; đáp án thường là số biên.

**Checklist trước khi submit:**
- [ ] `prev` khởi tạo bằng `'0'` (trạng thái ban đầu), không phải `target[0]`.
- [ ] So sánh ký tự với ký tự (`"0"`/`"1"`), hoặc số với số — không trộn `int` với `str`.
- [ ] Không lật thật hậu tố trong vòng lặp (tránh `O(n²)`).
- [ ] Thử nhanh: `"0" → 0`, `"1" → 1`, `"10" → 2`, `"101" → 3`, `"10111" → 3`.
- [ ] Tên method đúng `minFlips(self, target: str) -> int`.